**Imports**

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Subset
from torchvision import datasets,transforms
from sklearn.metrics import roc_auc_score

device = "mps" if torch.backends.mps.is_available() else "cpu"

**Basic Energy Formulations**

In [2]:
def energy_score(logits,T=1):
    return T * torch.logsumexp(logits/T,dim=1)

@torch.no_grad()
def get_energy_scores(model,loader,T=1,is_labelled=True):
    model.eval()
    scores = []
    for batch in loader:
        x = batch[0] if is_labelled else batch
        x = x.to(device)
        
        logits = model(x)
        scores.append(energy_score(logits,T=T).cpu())
    return torch.cat(scores)

def auroc(id_scores,ood_scores):
    y_true = [0]*len(id_scores)+[1]*len(ood_scores)        
    y_score = torch.cat([-id_scores,-ood_scores]).numpy()
    return roc_auc_score(y_true,y_score)

**Data**

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

import os
data_dir = os.path.expanduser('~/.pytorch/data') 

mnist_train  = datasets.MNIST(root=data_dir, train=True,  download=True, transform=transform)
mnist_test    = datasets.MNIST(root=data_dir, train=False, download=True, transform=transform)
fashion_train = datasets.FashionMNIST(root=data_dir, train=True,  download=True, transform=transform)
fashion_test  = datasets.FashionMNIST(root=data_dir, train=False, download=True, transform=transform)

id_labels = set(range(1, 10))
label_map = {orig: i for i, orig in enumerate(sorted(id_labels))}

def filter_indices(dataset, labels_to_keep):
    return [i for i, (_, y) in enumerate(dataset) if y in labels_to_keep]

class RemappedSubset(torch.utils.data.Dataset):
    def __init__(self, dataset, indices, label_map):
        self.dataset, self.indices, self.label_map = dataset, indices, label_map
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        x, y = self.dataset[self.indices[idx]]
        return x, self.label_map[y]

id_train_idx = filter_indices(mnist_train, id_labels)
id_train_set = RemappedSubset(mnist_train, id_train_idx, label_map)

oe_idx = torch.randperm(len(fashion_train))[:len(id_train_set)].tolist()
oe_set = Subset(fashion_train, oe_idx)

id_test_idx  = filter_indices(mnist_test, id_labels)
id_test_set  = RemappedSubset(mnist_test, id_test_idx, label_map)
zero_idx     = filter_indices(mnist_test, {0})

id_test_loader   = DataLoader(id_test_set, batch_size=256)
zero_loader      = DataLoader(Subset(mnist_test, zero_idx), batch_size=256)
fashion_loader   = DataLoader(fashion_test, batch_size=256) 
noise_dataset    = torch.randn(2000, 1, 28, 28)
noise_loader     = DataLoader(noise_dataset, batch_size=256)

**Model**

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

device = "mps" if torch.backends.mps.is_available() else "cpu"


class SmallCNN(nn.Module):
    def __init__(self, num_classes=9):
        super().__init__()
        self.conv1   = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2   = nn.Conv2d(32, 64, 3, padding=1)
        self.pool    = nn.MaxPool2d(2, 2)
        self.relu    = nn.ReLU()
        self.fc1     = nn.Linear(64 * 7 * 7, 128)
        self.fc2     = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.flatten(1)
        x = self.relu(self.fc1(x))
        return self.fc2(self.dropout(x))


def train_epoch_standard(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_accuracy(model,loader):
    model.eval()
    correct,total = 0,0
    
    for X,y in loader:
        X,y = X.to(device),y.to(device)
        correct += (model(X).argmax(1)==y).sum().item()
        total+=X.shape[0]
    
    return correct/total

model = SmallCNN(num_classes=9).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

id_loader = DataLoader(id_train_set, batch_size=128, shuffle=True)

print("=== Pretraining Base Model on MNIST (1-9) ===")
EPOCHS = 10
for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch_standard(model, id_loader, optimizer, criterion)
    test_acc = evaluate_accuracy(model, id_test_loader)
    print(f"Epoch {epoch+1:02d}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, test_acc={test_acc:.4f}")


=== Pretraining Base Model on MNIST (1-9) ===
Epoch 01: train_loss=0.2052, train_acc=0.9361, test_acc=0.9796
Epoch 02: train_loss=0.0616, train_acc=0.9810, test_acc=0.9869
Epoch 03: train_loss=0.0435, train_acc=0.9871, test_acc=0.9907
Epoch 04: train_loss=0.0341, train_acc=0.9898, test_acc=0.9912
Epoch 05: train_loss=0.0284, train_acc=0.9910, test_acc=0.9915
Epoch 06: train_loss=0.0235, train_acc=0.9926, test_acc=0.9919
Epoch 07: train_loss=0.0177, train_acc=0.9942, test_acc=0.9908
Epoch 08: train_loss=0.0171, train_acc=0.9946, test_acc=0.9906
Epoch 09: train_loss=0.0134, train_acc=0.9956, test_acc=0.9924
Epoch 10: train_loss=0.0110, train_acc=0.9964, test_acc=0.9919


**Inference Time Energy Score**

No retraining of model

In [6]:
print("---- 1 : Inference-time Energy Score (no retraining) ---\n")
for T in [1,10,100,1000]:
    id_s = get_energy_scores(model,id_test_loader,T=T)
    zero_s = get_energy_scores(model,zero_loader,T=T)
    print(f"T={T:4d} AUROC vs digit 0 : {auroc(id_s,zero_s):.4f}")
print()

T = 1
id_s      = get_energy_scores(model, id_test_loader,  T=T)
zero_s    = get_energy_scores(model, zero_loader,     T=T)
fashion_s = get_energy_scores(model, fashion_loader,  T=T)
noise_s   = get_energy_scores(model, noise_loader,    T=T, is_labelled=False)

print(f"Energy Score (T={T}) on pretrained model:")
print(f"  AUROC vs digit 0:      {auroc(id_s, zero_s):.4f}")
print(f"  AUROC vs FashionMNIST: {auroc(id_s, fashion_s):.4f}")
print(f"  AUROC vs noise:        {auroc(id_s, noise_s):.4f}")

@torch.no_grad()
def get_msp_scores(model,loader,is_labelled=True):
    model.eval()
    scores = []
    for batch in loader:
        x = batch[0] if is_labelled else batch
        x = x.to(device)
        probs = nn.Softmax(dim=1)(model(x))
        scores.append(probs.max(dim=1).values.cpu())
    return torch.cat(scores)

id_msp   = get_msp_scores(model, id_test_loader)
zero_msp = get_msp_scores(model, zero_loader)

print(f"\nMSP vs Energy on digit 0 (pretrained):")
print(f"  MSP    AUROC: {auroc(id_msp, zero_msp):.4f}")
print(f"  Energy AUROC: {auroc(id_s, zero_s):.4f}")
print(f"  Energy is >= MSP")

---- 1 : Inference-time Energy Score (no retraining) ---

T=   1 AUROC vs digit 0 : 0.9896
T=  10 AUROC vs digit 0 : 0.9607
T= 100 AUROC vs digit 0 : 0.0930
T=1000 AUROC vs digit 0 : 0.0787

Energy Score (T=1) on pretrained model:
  AUROC vs digit 0:      0.9896
  AUROC vs FashionMNIST: 0.9807
  AUROC vs noise:        1.0000

MSP vs Energy on digit 0 (pretrained):
  MSP    AUROC: 0.9813
  Energy AUROC: 0.9896
  Energy is >= MSP


**Energy Loss Definition**

In [ ]:
def energy_fine_tune_loss(logits_in,labels,logits_oe,m_in,m_out,lmbd=0.1,T=1):
    ce_loss = nn.CrossEntropyLoss()(logits_in,labels)
    
    E_in = -energy_score(logits_in,T)
    E_out = -energy_score(logits_oe,T=T)
    
    hinge_in = (nn.ReLU()(E_in-m_in)).pow(2).mean()
    hinge_out = (nn.ReLU()(m_out-E_out)).pow(2).mean()
    
    l_energy = hinge_in + hinge_out
    
    return ce_loss + lmbd*l_energy, ce_loss.item(), l_energy.item()

**Auxillary OOD Setup**

In [9]:
fashion_train = datasets.FashionMNIST(
    root="./data", train=True, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
)
oe_idx = torch.randperm(len(fashion_train))[:len(id_train_set)].tolist()
oe_set = Subset(fashion_train, oe_idx)

id_loader_ft = DataLoader(id_train_set, batch_size=128, shuffle=True)
oe_loader_ft = DataLoader(oe_set,       batch_size=128, shuffle=True)

**Fine-Tuning a new copy of model**

In [13]:
import copy
model_energy = copy.deepcopy(model)
optimizer_ft = torch.optim.Adam(model_energy.parameters(),lr=1e-3)

with torch.no_grad():
    sample_id_energies  = -get_energy_scores(model, id_test_loader)
    sample_ood_energies = -get_energy_scores(model, fashion_loader)
    print(f"\nPretrained model energy stats:")
    print(f"  ID  energy: mean={sample_id_energies.mean():.2f}, std={sample_id_energies.std():.2f}")
    print(f"  OOD energy: mean={sample_ood_energies.mean():.2f}, std={sample_ood_energies.std():.2f}")

m_in = -17
m_out = -6
lmbd = 0.1

def train_epoch_energy(model,id_loader,oe_loader,optimizer,m_in,m_out,lmbd,T=1.0):
    model.train()
    total_loss,total_ce,total_energy,correct,total = 0,0,0,0,0
    oe_iter = iter(oe_loader)

    for x_in,y_in in id_loader:
        x_in,y_in = x_in.to(device),y_in.to(device)

        try:
            x_oe,y_oe = next(oe_iter)
        except StopIteration:
            oe_iter  = iter(oe_loader)
            x_oe,y_oe = next(oe_iter)
        x_oe = x_oe.to(device)

        optimizer.zero_grad()

        logits_in  = model(x_in)
        logits_oe  = model(x_oe)

        loss, ce,l_e = energy_fine_tune_loss(
            logits_in,y_in,logits_oe,m_in,m_out,lmbd=lmbd,T=T
        )
        loss.backward()
        optimizer.step()

        total_loss += loss.item()*x_in.size(0)
        total_ce += ce*x_in.size(0)
        total_energy += l_e*x_in.size(0)
        correct += (logits_in.argmax(1)==y_in).sum().item()
        total += x_in.size(0)

    n = total
    return total_loss/n,total_ce/n,total_energy/n,correct/n


@torch.no_grad()
def evaluate_accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    for x,y in loader:
        x,y = x.to(device),y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total += x.size(0)
    return correct / total


Pretrained model energy stats:
  ID  energy: mean=-17.80, std=5.01
  OOD energy: mean=-5.50, std=2.98


In [14]:
print("\n --- Part 2: Energy Fine-tuning ---\n")
EPOCHS = 10
for epoch in range(EPOCHS):
    loss,ce,l_e,acc = train_epoch_energy(
        model_energy,id_loader_ft,oe_loader_ft,
        optimizer_ft,m_in=m_in,m_out=m_out,lmbd=lmbd
    )
    id_acc = evaluate_accuracy(model_energy,id_test_loader)
    print(f"Epoch {epoch+1}: loss={loss:.4f} (ce={ce:.4f}, energy={l_e:.4f}), "
          f"train_acc={acc:.4f}, id_test_acc={id_acc:.4f}")
    
print("\n --- Results after Energy Fine-tuning ---\n")
id_s_ft = get_energy_scores(model_energy,id_test_loader)
zero_s_ft = get_energy_scores(model_energy,zero_loader)
fashion_s_ft = get_energy_scores(model_energy,fashion_loader)
noise_s_ft = get_energy_scores(model_energy,noise_loader,is_labelled=False)

print(f"Energy Fine-tuned (m_in={m_in},m_out={m_out},lmbd={lmbd}):")
print(f"AUROC vs digit 0: {auroc(id_s_ft,zero_s_ft):.4f}")
print(f"AUROC vs FashionMNIST: {auroc(id_s_ft,fashion_s_ft):.4f}")
print(f"AUROC vs noise: {auroc(id_s_ft,noise_s_ft):.4f}")


 --- Part 2: Energy Fine-tuning ---

Epoch 1: loss=0.0623 (ce=0.0196, energy=0.4269), train_acc=0.9943, id_test_acc=0.9925
Epoch 2: loss=0.0188 (ce=0.0131, energy=0.0570), train_acc=0.9960, id_test_acc=0.9935
Epoch 3: loss=0.0145 (ce=0.0113, energy=0.0322), train_acc=0.9963, id_test_acc=0.9914
Epoch 4: loss=0.0118 (ce=0.0101, energy=0.0174), train_acc=0.9968, id_test_acc=0.9918
Epoch 5: loss=0.0139 (ce=0.0119, energy=0.0197), train_acc=0.9961, id_test_acc=0.9925
Epoch 6: loss=0.0109 (ce=0.0094, energy=0.0150), train_acc=0.9969, id_test_acc=0.9920
Epoch 7: loss=0.0090 (ce=0.0079, energy=0.0114), train_acc=0.9976, id_test_acc=0.9917
Epoch 8: loss=0.0119 (ce=0.0108, energy=0.0106), train_acc=0.9963, id_test_acc=0.9912
Epoch 9: loss=0.0111 (ce=0.0103, energy=0.0080), train_acc=0.9967, id_test_acc=0.9916
Epoch 10: loss=0.0096 (ce=0.0088, energy=0.0084), train_acc=0.9971, id_test_acc=0.9926

 --- Results after Energy Fine-tuning ---

Energy Fine-tuned (m_in=-17,m_out=-6,lmbd=0.1):
AUROC vs 